# Ноутбук с графиками PDP и ICE

## 1. Импорт бибилиотек и конфигурация проекта

In [5]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import Ridge
from category_encoders.cat_boost import CatBoostEncoder
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV, KFold, RandomizedSearchCV
import copy
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.model_selection import cross_validate
import xgboost as xgb
import pyarrow
import phik
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
import catboost
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate
)
from catboost import CatBoostRegressor
import mlflow.sklearn
from phik.report import plot_correlation_matrix
import plotly
import mlflow
import time
from scipy.stats import randint, uniform, loguniform
import os
from datetime import datetime
import category_encoders as ce
import joblib
from sklearn.inspection import PartialDependenceDisplay

In [3]:
# Создадим словарь конфигураций.

CONFIG = {
    # Константы
    "DEV_MODE": False,
    "DEV_SAMPLE_SIZE": 100000,
    "RANDOM_STATE": 42,
    # Целевая переменная 
    "TARGET": "Цена",
    "YEAR": datetime.now().year
}

In [4]:
train = pd.read_parquet("../data/optimized/train_optimized.parquet")
test = pd.read_parquet("../data/optimized/test_optimized.parquet")

In [9]:
X_train = train.drop(columns=[CONFIG["TARGET"]])
y_train = train[CONFIG["TARGET"]]
X_test = test.drop(columns=[CONFIG["TARGET"]])
y_test = test[CONFIG["TARGET"]]

dir = 'C:/project/car-price-analyzer/src/mlflow_runs'
os.makedirs(dir, exist_ok=True)
mlflow.set_tracking_uri(f'sqlite:///{dir}/mlflow.db')
mlflow.set_experiment('ice_pdp_analysis')

2026/08/05 14:33:52 INFO mlflow.tracking.fluent: Experiment with name 'ice_pdp_analysis' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:c:/project/car-price-analyzer/researches/mlruns/6', creation_time=1785929632723, effective_trace_archival_retention=None, experiment_id='6', last_update_time=1785929632723, lifecycle_stage='active', name='ice_pdp_analysis', tags={}, trace_location=None, workspace='default'>

## 2. Построение графиков

In [12]:
params = {
    'bootstrap_type': 'Bernoulli',
    'iterations': 2500,
    'learning_rate': 0.042248138023520114,
    'depth': 10,
    'l2_leaf_reg': 9.394802632887766,
    'random_strength': 0.8303775333704183,
    'border_count': 207,
    'subsample': 0.5483853226992782,
    'loss_function': 'MAE',
    'allow_writing_files': False,
    'eval_metric': 'MAE',
    'random_seed': 42,
    'thread_count': -1,
    'task_type': 'GPU',
    'verbose': 500
}
text_features = ['Комплектация', 'Название машины']

cat_features = [
    col for col in X_train.select_dtypes(include=['object', 'category']).columns 
    if col not in text_features
]

for col in text_features + cat_features:
    X_train[col] = X_train[col].astype(object).fillna('Unknown').astype(str)
    if col in X_test.columns:
        X_test[col] = X_test[col].astype(object).fillna('Unknown').astype(str)
model = CatBoostRegressor(**params)
wrapped_model = TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1
    )
wrapped_model.fit(X_train, y_train, cat_features=cat_features, text_features=text_features)

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.6902084	total: 407ms	remaining: 16m 56s
500:	learn: 0.1563041	total: 46.4s	remaining: 3m 5s
1000:	learn: 0.1491160	total: 1m 20s	remaining: 2m 1s
1500:	learn: 0.1456508	total: 1m 55s	remaining: 1m 16s
2000:	learn: 0.1422274	total: 2m 37s	remaining: 39.2s
2499:	learn: 0.1395371	total: 3m 18s	remaining: 0us


,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.","CatBoostRegre..., verbose=500)"
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,27
regressor_ regressor_: objectFitted regressor.,CatBoostRegressor,"CatBoostRegre..., verbose=500)"
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,FunctionTransformer,FunctionTrans...validate=True)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay

features_to_plot = ['Год', 'Пробег', 'Мощность']

# Строим PDP и ICE вместе
fig, ax = plt.subplots(nrows=3, ncols=1, figsize=(12, 16), dpi=150)

PartialDependenceDisplay.from_estimator(
    estimator=wrapped_model,
    X=X_train,
    features=features_to_plot,
    kind='both',
    subsample=50, # Ограничим число ICE-линий до 50, чтобы график не превратился в кашу
    ax=ax,
    n_jobs=-1
)

plt.suptitle('PDP и ICE графики для модели оценки авто', fontsize=16)
# plt.tight_layout()
plt.show()